# Weatherstack — Weather Data

Demonstrates **weatherstack_current**, **weatherstack_forecast**, **weatherstack_historical**, and **weatherstack_autocomplete** tools
using the [Weatherstack API](https://weatherstack.com).

Requires a `WEATHERSTACK_ACCESS_KEY` environment variable set in `.env`.

---
## Setup

In [ ]:
!pip install -q -e "../.."
!pip install -q -r ../requirements.txt

In [ ]:
from dotenv import load_dotenv

load_dotenv()

from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

from demos.shared.llm_setup import get_llm
from pymcpx.weatherstack import (
    WeatherstackAutocompleteTool,
    WeatherstackCurrentTool,
    WeatherstackForecastTool,
    WeatherstackHistoricalTool,
)

llm = get_llm()

tools = [
    WeatherstackCurrentTool(),
    WeatherstackForecastTool(),
    WeatherstackHistoricalTool(),
    WeatherstackAutocompleteTool(),
]

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a weather data assistant. Use the Weatherstack tools to "
            "answer questions about current weather, forecasts, and historical data.\n\n"
            "Available tools:\n"
            "- weatherstack_current(query, units) \u2014 current weather for any location\n"
            "- weatherstack_forecast(query, forecast_days) \u2014 weather forecast (up to 14 days)\n"
            "- weatherstack_historical(query, historical_date) \u2014 historical weather for a date\n"
            "- weatherstack_autocomplete(query) \u2014 search for locations by partial name",
        ),
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

agent = create_tool_calling_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
print("\u2705 Agent ready. Running scenarios...")

### Scenario 1: Current Weather

Ask for current conditions in a major city.

In [ ]:
result = executor.invoke({"input": "What is the current weather in New York? Use metric units."})
print(result["output"])

### Scenario 2: Weather Forecast

Get a multi-day forecast for a location.

In [ ]:
result = executor.invoke({"input": "What is the 5-day weather forecast for London, UK?"})
print(result["output"])

### Scenario 3: Historical Weather

Look up past weather data for a specific date.

In [ ]:
result = executor.invoke(
    {"input": "What was the weather in Tokyo on 2025-06-01? Use metric units."}
)
print(result["output"])

### Scenario 4: Location Autocomplete

Search for a city by partial name to find the correct location identifier.

In [ ]:
result = executor.invoke(
    {"input": "Search for cities matching 'San Fran' to find the correct location name."}
)
print(result["output"])